# Plots for pure symmetrization in SALNet (fig. 9)

In this experiment, we investigate the convergence speed of SAL alone in the SALNet, when the forward weights are constant.
This helps us to determine the optimal tradeoff between fast convergence and low alignment angles by choosing an appropriate learning rate.

The raw data is simulated by `salnet_symm.py` and stored in `../../results/symm_net/puresymm`. 

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.ticker as ticker
import matplotlib.pyplot as plt

In [ ]:
SAVEFIG = True

# Point this at the sweep directory produced by sweep_symm.py
SWEEP_DIR = Path("../../results/symm_net/puresymm/puresymm")

FIG_DIR = Path("../../figs/symm_net")
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Learning rates — must match what was passed to sweep_symm.py
LRS = [0.01, 0.02, 0.04, 0.08]

In [ ]:
# define the style etc.
mpl.style.use("../../mystyle.mpl")
plt.style.use("tableau-colorblind10")

In [ ]:
def load_runs(sweep_dir: Path, lr: float, metric_key: str) -> pd.DataFrame:
    """Load a metric time series for all seeds of one learning rate.

    Args:
        sweep_dir: Root sweep directory (contains lr_* sub-dirs).
        lr: Learning rate value.
        metric_key: Key inside metrics["scalars"] (e.g. "symm/angle/0").

    Returns:
        DataFrame with one column per seed (seed_0, seed_1, …) and one row
        per epoch.
    """
    df = pd.DataFrame()
    lr_dir = sweep_dir / f"lr_{lr}"
    for seed_dir in sorted(lr_dir.glob("seed_*")):
        path = seed_dir / "metrics.json"
        if not path.exists():
            continue
        with open(path) as f:
            data = json.load(f)
        df[seed_dir.name] = data["scalars"].get(metric_key, [])
    df_stats = pd.DataFrame({"mean": df.mean(axis=1), "std": df.std(axis=1)})
    return df_stats


def load_params(sweep_dir: Path, lr: float) -> dict:
    """Load params dict from the first available seed of a learning rate."""
    lr_dir = sweep_dir / f"lr_{lr}"
    for seed_dir in sorted(lr_dir.glob("seed_*")):
        path = seed_dir / "metrics.json"
        if path.exists():
            with open(path) as f:
                return json.load(f)["params"]
    return {}

In [ ]:
dfs_1, dfs_2, dfs_3 = [], [], []
for lr in LRS:
    n_seeds = len(list((SWEEP_DIR / f"lr_{lr}").glob("seed_*")))
    dfs_1.append(load_runs(SWEEP_DIR, lr, "symm/angle/0")["mean"])
    dfs_2.append(load_runs(SWEEP_DIR, lr, "symm/angle/1")["mean"])
    dfs_3.append(load_runs(SWEEP_DIR, lr, "symm/angle/2")["mean"])

df_1 = pd.concat(dfs_1, axis=1)
df_1.columns = LRS
df_2 = pd.concat(dfs_2, axis=1)
df_2.columns = LRS
df_3 = pd.concat(dfs_3, axis=1)
df_3.columns = LRS


params0 = load_params(SWEEP_DIR, LRS[0])
t_ref = 0.01  # seconds (t_ref=10 timesteps × 1 ms/timestep)
t_iter = t_ref * params0["batchsize"] * params0["len_epoch"]
t_max = (params0["n_epochs"] + 1) * t_iter

In [ ]:
MAX_T_ID = 64 * 30
num_epochs = 200
sec_per_epoch = 320
upper_axis_color = "red"
ts = np.arange(0, t_max, t_iter)

fig, ax = plt.subplots(
    1, 3, figsize=(12 / 2.54, 5.5 / 2.54), sharey=True
)  # Reduced figsize

# 1. Plotting
lines = ax[0].plot(
    ts[:MAX_T_ID], df_1.iloc[:MAX_T_ID, :], label=[f"lr={lr}" for lr in LRS]
)
ax[1].plot(ts[:MAX_T_ID], df_2.iloc[:MAX_T_ID, :])
ax[2].plot(ts[:MAX_T_ID], df_3.iloc[:MAX_T_ID, :])

# Configure Left Panel Y-Axis
# USE THE FOLLOWING LINE IF YOU HAVE LATEX INSTALLED:
ax[0].set_ylabel(r"$\measuredangle (\mathbf W^T, \mathbf B)\ [\mathrm{deg}]$")
# If not, this one:
ax[0].set_ylabel(r"angle $( W^T, B)$ [deg]")
ax[0].set_ylim(bottom=0.0)


# Define a custom formatter for scientific notation on the tick itself
# e.g. 100000 -> 1x10^5 or 10^5
def scientific_notation_formatter(x, pos):
    if x == 0:
        return "0"
    # Format as 1e5, 2e5, etc.
    s = "{:.0e}".format(x)
    # Convert '1e+05' to latex '10^5' or '2 \cdot 10^5'
    base, exponent = s.split("e")
    exponent = int(exponent)
    if base == "1":
        return r"$10^{%d}$" % exponent
    else:
        return r"$%s \cdot 10^{%d}$" % (base, exponent)


for i in range(3):
    # Apply the custom formatter to the primary x-axis
    ax[i].xaxis.set_major_formatter(ticker.FuncFormatter(scientific_notation_formatter))

    ax[i].minorticks_on()
    ax[i].grid(which="major")

    # --- SECONDARY AXIS ---
    secax = ax[i].secondary_xaxis(
        "top", functions=(lambda t: t / sec_per_epoch, lambda e: e * sec_per_epoch)
    )
    secax.set_xticks(np.arange(0, num_epochs + 1, 100))
    ax[i].axvline(
        num_epochs * sec_per_epoch,
        linestyle="--",
        linewidth=0.8,
        color=upper_axis_color,
    )

    if i == 1:
        secax.set_xlabel("Epochs", color=upper_axis_color)

    secax.tick_params(axis="x", colors=upper_axis_color)
    secax.spines["top"].set_color(upper_axis_color)

    # Internal Titles
    ax[i].text(
        0.95,
        0.95,
        f"FC {i+1}",
        transform=ax[i].transAxes,
        horizontalalignment="right",
        verticalalignment="top",
        bbox=dict(facecolor="white", alpha=0.6, edgecolor="none", pad=2),
    )

ax[1].set_xlabel(r"$t$ [s]")

# --- FIGURE LEVEL LEGEND ---
handles, labels = ax[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.07),
    ncol=len(labels),
    frameon=True,
)

plt.tight_layout()

In [ ]:
fig.savefig(FIG_DIR / "salnet_time.png", dpi=300)
fig.savefig(FIG_DIR / "salnet_time.pdf")
fig.savefig(FIG_DIR / "salnet_time.svg")